**PALEOS MgSiO₃ Equation of State: Lookup Table Generation**

This notebook generates pre-computed thermodynamic property tables for MgSiO₃ across the complete
planetary mantle phase diagram. The tables are designed for efficient interpolation in planetary
interior models.

**Contents:**
1. **Grid Resolution Optimization** — Determines the minimum grid resolution (points per decade)
   required to achieve a target interpolation accuracy.
2. **Table Generation** — Creates the final lookup table with all seven thermodynamic properties
   plus phase information.

**Author:** Mara Attia  
**Date:** March 2026  

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import RegularGridInterpolator
import warnings
import datetime

# Import PALEOS MgSiO₃ EoS module
from paleos import mgsio3_eos as mgsio3

# For reproducibility
np.random.seed(42)

# Pre-initialize one EoS instance per phase to avoid repeated
# construction overhead (Wolf18/RTpress init is expensive).
_eos_instances = {
    'solid-en':     mgsio3.Sokolova22(phase='orthoen'),
    'solid-lpcen':  mgsio3.Sokolova22(phase='lp-cen'),
    'solid-hpcen':  mgsio3.Sokolova22(phase='hp-cen'),
    'solid-brg':    mgsio3.Wolf15(x_Fe=0.0),
    'solid-ppv':    mgsio3.Sakai16(),
    'liquid':       mgsio3.Wolf18(),
}

print("PALEOS MgSiO₃ EoS Lookup Table Generator")
print("=======================================")

---
# Grid Resolution Optimization

We determine the optimal grid resolution by:
1. Creating log-uniform grids with varying resolution (points per decade)
2. Evaluating density on each grid
3. Drawing a fixed-size control sample from the P-T domain
4. Computing relative interpolation error for each control point
5. Tracking median and 99th percentile errors as a function of resolution

**Key considerations:**
- Resolution is specified as **points per decade** (ppd), which naturally accounts for
  the different dynamic ranges of P (~ 8 decades) and T (~ 2.5 decades)
- Interpolation is performed in (log P, log T) space for accuracy
- We track both median (typical error) and 99th percentile (worst-case near boundaries)
- We also verify that the chosen resolution is adequate for thermal expansion α, as it has the sharpest features across phase space

In [ ]:
# === FIXED PARAMETERS FOR RESOLUTION TEST ===

# P-T domain (same as final table)
P_MIN = 1e5      # Pa (1 bar)
P_MAX = 1e14     # Pa (100 TPa)
T_MIN = 300.0    # K
T_MAX = 1e5      # K (100,000 K)

# Compute number of decades
N_DECADES_P = np.log10(P_MAX) - np.log10(P_MIN)  # 8 decades
N_DECADES_T = np.log10(T_MAX) - np.log10(T_MIN)  # ~2.52 decades

# Control sample size (fixed for all resolutions)
N_CONTROL = 5000

# Points per decade to test
PPD_VALUES = [20, 40, 60, 80, 100, 150, 300]

# Target precision threshold
TARGET_PRECISION = 1e-4  # relative error

print(f"P range: [{P_MIN:.0e}, {P_MAX:.0e}] Pa  ({N_DECADES_P:.1f} decades)")
print(f"T range: [{T_MIN:.0f}, {T_MAX:.0e}] K  ({N_DECADES_T:.2f} decades)")
print(f"Control sample size: {N_CONTROL}")
print(f"Points per decade to test: {PPD_VALUES}")
print(f"Target precision: {TARGET_PRECISION:.0e}")

In [ ]:
def make_log_grid(x_min, x_max, ppd):
    """
    Create a log-uniform grid with specified points per decade.
    
    Parameters
    ----------
    x_min : float
        Minimum value (linear scale)
    x_max : float
        Maximum value (linear scale)
    ppd : int
        Points per decade
    
    Returns
    -------
    log_x : array
        log10(x) grid values
    n_points : int
        Total number of points
    """
    log_min = np.log10(x_min)
    log_max = np.log10(x_max)
    n_decades = log_max - log_min
    n_points = int(np.ceil(n_decades * ppd)) + 1
    log_x = np.linspace(log_min, log_max, n_points)
    return log_x, n_points


def evaluate_property_on_grid(log_P_grid, log_T_grid, property_func):
    """
    Evaluate a thermodynamic property on a 2D grid.
    
    Parameters
    ----------
    log_P_grid : array
        1D array of log10(P) values
    log_T_grid : array
        1D array of log10(T) values
    property_func : callable
        Function that takes (P, T) and returns property value.
        Should handle phase selection internally.
    
    Returns
    -------
    values : 2D array
        Property values, shape (len(log_P_grid), len(log_T_grid))
    """
    nP = len(log_P_grid)
    nT = len(log_T_grid)
    values = np.zeros((nP, nT))
    
    for i, log_P in enumerate(log_P_grid):
        P = 10**log_P
        for j, log_T in enumerate(log_T_grid):
            T = 10**log_T
            try:
                values[i, j] = property_func(P, T)
            except Exception:
                values[i, j] = np.nan
    
    return values


def get_eos_for_PT(P, T):
    """Phase determination + pre-initialised EoS lookup."""
    phase = mgsio3.get_mgsio3_phase(P, T)
    return _eos_instances[phase], phase

def get_density(P, T):
    """Get density using appropriate EoS for given P-T."""
    eos, _ = get_eos_for_PT(P, T)
    return eos.density(P, T)


def get_thermal_expansion(P, T):
    """Get thermal expansion using appropriate EoS for given P-T."""
    eos, _ = get_eos_for_PT(P, T)
    return eos.thermal_expansion(P, T)

In [ ]:
def generate_control_sample(n_points, P_min, P_max, T_min, T_max, seed=None):
    """
    Generate log-uniform control sample in P-T space.
    
    Points are drawn from continuous distributions to avoid
    landing exactly on grid nodes.
    """
    if seed is not None:
        np.random.seed(seed)
    
    log_P = np.random.uniform(np.log10(P_min), np.log10(P_max), n_points)
    log_T = np.random.uniform(np.log10(T_min), np.log10(T_max), n_points)
    
    return log_P, log_T


def compute_interpolation_errors(log_P_grid, log_T_grid, grid_values,
                                  control_log_P, control_log_T,
                                  true_func):
    """
    Compute relative interpolation errors for control sample.
    
    Interpolation is performed in (log P, log T) space using
    bilinear interpolation.
    
    Returns
    -------
    rel_errors : array
        Relative errors |interp - true| / |true| for each control point
    phase_crossing : array
        Boolean array indicating if control point crosses a phase boundary
        when interpolated (i.e., nearby grid points are in different phases)
    """
    # Create interpolator in log-log space
    interpolator = RegularGridInterpolator(
        (log_P_grid, log_T_grid), 
        grid_values,
        method='linear',
        bounds_error=False,
        fill_value=np.nan
    )
    
    n_points = len(control_log_P)
    rel_errors = np.zeros(n_points)
    phase_crossing = np.zeros(n_points, dtype=bool)
    
    for k in range(n_points):
        log_P = control_log_P[k]
        log_T = control_log_T[k]
        P = 10**log_P
        T = 10**log_T
        
        # Interpolated value
        interp_val = interpolator([[log_P, log_T]])[0]
        
        # True value
        try:
            true_val = true_func(P, T)
        except Exception:
            rel_errors[k] = np.nan
            continue
        
        # Relative error
        if np.isnan(interp_val) or np.isnan(true_val) or abs(true_val) < 1e-30:
            rel_errors[k] = np.nan
        else:
            rel_errors[k] = abs(interp_val - true_val) / abs(true_val)
        
        # Check for phase crossing (find neighboring grid points)
        i_low = np.searchsorted(log_P_grid, log_P) - 1
        j_low = np.searchsorted(log_T_grid, log_T) - 1
        i_low = max(0, min(i_low, len(log_P_grid) - 2))
        j_low = max(0, min(j_low, len(log_T_grid) - 2))
        
        # Get phases at 4 corners
        corner_phases = set()
        for di in [0, 1]:
            for dj in [0, 1]:
                P_corner = 10**log_P_grid[i_low + di]
                T_corner = 10**log_T_grid[j_low + dj]
                corner_phases.add(mgsio3.get_mgsio3_phase(P_corner, T_corner))
        
        phase_crossing[k] = len(corner_phases) > 1
    
    return rel_errors, phase_crossing

In [ ]:
# Generate fixed control sample (same for all resolutions)
print("Generating control sample...")
control_log_P, control_log_T = generate_control_sample(
    N_CONTROL, P_MIN, P_MAX, T_MIN, T_MAX, seed=42
)
print(f"  Generated {N_CONTROL} control points")

In [ ]:
# Test different resolutions for DENSITY
print("\nTesting grid resolutions for density...")
print("-" * 80)
print(f"{'ppd':>5s} {'n_P':>6s} {'n_T':>6s} {'total':>10s} {'median':>12s} {'p99':>12s} {'max':>12s} \
{'phase_x':>10s}")
print("-" * 80)

results_density = []

for ppd in PPD_VALUES:
    # Create grid with this points-per-decade
    log_P_grid, n_P = make_log_grid(P_MIN, P_MAX, ppd)
    log_T_grid, n_T = make_log_grid(T_MIN, T_MAX, ppd)
    total_points = n_P * n_T
    
    # Evaluate density on grid
    rho_grid = evaluate_property_on_grid(log_P_grid, log_T_grid, get_density)
    
    # Compute errors
    errors, phase_cross = compute_interpolation_errors(
        log_P_grid, log_T_grid, rho_grid,
        control_log_P, control_log_T,
        get_density
    )
    
    # Filter valid errors
    valid_errors = errors[~np.isnan(errors)]
    
    # Statistics
    median_err = np.median(valid_errors)
    p99_err = np.percentile(valid_errors, 99)
    max_err = np.max(valid_errors)
    n_phase_cross = np.sum(phase_cross)
    frac_phase_cross = n_phase_cross / N_CONTROL
    
    results_density.append({
        'ppd': ppd,
        'n_P': n_P,
        'n_T': n_T,
        'total': total_points,
        'median': median_err,
        'p99': p99_err,
        'max': max_err,
        'phase_cross_frac': frac_phase_cross
    })
    
    print(f"{ppd:5d} {n_P:6d} {n_T:6d} {total_points:10,d} {median_err:12.2e} {p99_err:12.2e} \
    {max_err:12.2e} {frac_phase_cross:10.1%}")

print("-" * 80)

In [ ]:
# Test different resolutions for THERMAL EXPANSION (secondary check)
print("\nTesting grid resolutions for thermal expansion (secondary check)...")
print("-" * 60)
print(f"{'ppd':>5s} {'n_P':>6s} {'n_T':>6s} {'median':>12s} {'p99':>12s}")
print("-" * 60)

results_alpha = []

# Test a subset of resolutions for alpha
alpha_ppd_values = [ppd for ppd in PPD_VALUES if ppd >= 60]

for ppd in alpha_ppd_values:
    # Create grid
    log_P_grid, n_P = make_log_grid(P_MIN, P_MAX, ppd)
    log_T_grid, n_T = make_log_grid(T_MIN, T_MAX, ppd)
    
    # Evaluate alpha on grid (with warning suppression for edge cases)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        alpha_grid = evaluate_property_on_grid(log_P_grid, log_T_grid, get_thermal_expansion)
    
    # Compute errors
    errors, _ = compute_interpolation_errors(
        log_P_grid, log_T_grid, alpha_grid,
        control_log_P, control_log_T,
        get_thermal_expansion
    )
    
    # Filter valid errors
    valid_errors = errors[~np.isnan(errors)]
    
    # Statistics
    median_err = np.median(valid_errors)
    p99_err = np.percentile(valid_errors, 99)
    
    results_alpha.append({
        'ppd': ppd,
        'n_P': n_P,
        'n_T': n_T,
        'median': median_err,
        'p99': p99_err
    })
    
    print(f"{ppd:5d} {n_P:6d} {n_T:6d} {median_err:12.2e} {p99_err:12.2e}")

print("-" * 60)

In [ ]:
# Diagnostic plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=300)

# Left panel: Density errors
ax1 = axes[0]
ppds = [r['ppd'] for r in results_density]
medians = [r['median'] for r in results_density]
p99s = [r['p99'] for r in results_density]

ax1.loglog(ppds, medians, 'o-', color='C0', label='Median', markersize=8)
ax1.loglog(ppds, p99s, 's--', color='C1', label='99th percentile', markersize=8)
ax1.axhline(TARGET_PRECISION, color='k', linestyle=':', linewidth=2, 
            label=f'Target')

ax1.set_xlabel('Points per decade', fontsize=12)
ax1.set_ylabel(r'$|\Delta\rho|/\rho$', fontsize=12)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Right panel: Thermal expansion errors
ax2 = axes[1]
ppds_alpha = [r['ppd'] for r in results_alpha]
medians_alpha = [r['median'] for r in results_alpha]
p99s_alpha = [r['p99'] for r in results_alpha]

ax2.loglog(ppds_alpha, medians_alpha, 'o-', color='C0', label='Median', markersize=8)
ax2.loglog(ppds_alpha, p99s_alpha, 's--', color='C1', label='99th percentile', markersize=8)
ax2.axhline(TARGET_PRECISION, color='k', linestyle=':', linewidth=2,
            label=f'Target')

ax2.set_xlabel('Points per decade', fontsize=12)
ax2.set_ylabel(r'$|\Delta\alpha|/\alpha$', fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

In [ ]:
# Determine optimal resolution
def find_optimal_ppd(results, target, metric='median'):
    """Find minimum points-per-decade achieving target precision."""
    for r in results:
        if r[metric] <= target:
            return r['ppd']
    return results[-1]['ppd']  # Return largest if none meet target

optimal_ppd_median = find_optimal_ppd(results_density, TARGET_PRECISION, 'median')
optimal_ppd_p99 = find_optimal_ppd(results_density, TARGET_PRECISION, 'p99')

# Get grid sizes for recommended resolution
_, rec_n_P = make_log_grid(P_MIN, P_MAX, optimal_ppd_p99)
_, rec_n_T = make_log_grid(T_MIN, T_MAX, optimal_ppd_p99)

print("\n" + "=" * 70)
print("RESOLUTION OPTIMIZATION SUMMARY")
print("=" * 70)
print(f"Target precision: {TARGET_PRECISION:.0e}")
print(f"\nMinimum ppd for median(Δρ/ρ) ≤ target:  {optimal_ppd_median} ppd")
print(f"Minimum ppd for p99(Δρ/ρ) ≤ target:     {optimal_ppd_p99} ppd")
print(f"\nRecommended resolution: {optimal_ppd_p99} points per decade")
print(f"  → P grid: {rec_n_P} points ({N_DECADES_P:.1f} decades)")
print(f"  → T grid: {rec_n_T} points ({N_DECADES_T:.2f} decades)")
print(f"  → Total:  {rec_n_P * rec_n_T:,} points")
print("  (based on 99th percentile to ensure good accuracy near phase boundaries)")
print("=" * 70)

---
# Lookup Table Generation

This section generates the final lookup table with:
- All seven thermodynamic properties
- Phase identification for each grid point
- Comprehensive header with usage information

In [ ]:
# ============================================================================
# USER INPUT CELL - Modify these parameters as needed
# ============================================================================

# Pressure range [Pa]
TABLE_P_MIN = 1e5       # 1 bar
TABLE_P_MAX = 1e14      # 100 TPa

# Temperature range [K]
TABLE_T_MIN = 300.0     # 300 K
TABLE_T_MAX = 1e5       # 100,000 K

# Grid resolution: points per decade (use value from Section 1, or override)
TABLE_PPD = 600         # Points per decade

# Output filename
OUTPUT_FILENAME = "paleos_mgsio3_eos_table_pt_highres.dat"

# ============================================================================
# Compute actual grid sizes from ppd
log_P_table, TABLE_N_P = make_log_grid(TABLE_P_MIN, TABLE_P_MAX, TABLE_PPD)
log_T_table, TABLE_N_T = make_log_grid(TABLE_T_MIN, TABLE_T_MAX, TABLE_PPD)
P_table = 10**log_P_table
T_table = 10**log_T_table

n_decades_P = np.log10(TABLE_P_MAX) - np.log10(TABLE_P_MIN)
n_decades_T = np.log10(TABLE_T_MAX) - np.log10(TABLE_T_MIN)

print("Table generation parameters:")
print(f"  P range: [{TABLE_P_MIN:.0e}, {TABLE_P_MAX:.0e}] Pa  ({n_decades_P:.1f} decades)")
print(f"  T range: [{TABLE_T_MIN:.0f}, {TABLE_T_MAX:.0e}] K  ({n_decades_T:.2f} decades)")
print(f"  Resolution: {TABLE_PPD} points per decade")
print(f"  Grid size: {TABLE_N_P} x {TABLE_N_T} = {TABLE_N_P * TABLE_N_T:,} points")
print(f"  Output: {OUTPUT_FILENAME}")

In [ ]:
def phase_to_string(phase):
    """
    Convert phase name to descriptive string for table.
    """
    phase_map = {
        'solid-en': 'solid-en',
        'solid-lpcen': 'solid-lpcen',
        'solid-hpcen': 'solid-hpcen',
        'solid-brg': 'solid-brg',
        'solid-ppv': 'solid-ppv',
        'liquid': 'liquid'
    }
    return phase_map.get(phase, 'unknown')


def get_eos_for_PT(P, T):
    """Phase determination + pre-initialised EoS lookup."""
    phase = mgsio3.get_mgsio3_phase(P, T)
    return _eos_instances[phase], phase


def compute_all_properties(P, T):
    """
    Compute all thermodynamic properties at a single (P, T) point.
    
    Returns
    -------
    dict with keys:
        'rho': density [kg/m³]
        'u': specific internal energy [J/kg]
        's': specific entropy [J/(kg·K)]
        'cp': isobaric heat capacity [J/(kg·K)]
        'cv': isochoric heat capacity [J/(kg·K)]
        'alpha': thermal expansion [K⁻¹]
        'nabla_ad': adiabatic gradient [dimensionless]
        'phase': phase string
    """
    eos, phase = get_eos_for_PT(P, T)
    
    try:
        rho = eos.density(P, T)
    except:
        rho = np.nan
    
    try:
        u = eos.specific_internal_energy(P, T)
    except:
        u = np.nan
    
    try:
        s = eos.specific_entropy(P, T)
    except:
        s = np.nan
    
    try:
        cp = eos.isobaric_heat_capacity(P, T)
    except:
        cp = np.nan
    
    try:
        cv = eos.isochoric_heat_capacity(P, T)
    except:
        cv = np.nan
    
    try:
        alpha = eos.thermal_expansion(P, T)
    except:
        alpha = np.nan
    
    try:
        nabla_ad = eos.adiabatic_gradient(P, T)
    except:
        nabla_ad = np.nan
    
    return {
        'rho': rho,
        'u': u,
        's': s,
        'cp': cp,
        'cv': cv,
        'alpha': alpha,
        'nabla_ad': nabla_ad,
        'phase': phase_to_string(phase)
    }

In [ ]:
# Generate the table
print("Generating lookup table...")
print("=" * 60)

# Storage for all data
table_data = []

total_points = TABLE_N_P * TABLE_N_T
n_computed = 0
n_failed = 0

print(f"Computing {total_points:,} grid points...")

for i, P in enumerate(P_table):
    if (i + 1) % 50 == 0 or i == 0:
        print(f"  Progress: {i+1}/{TABLE_N_P} pressure slices "
              f"({100*(i+1)/TABLE_N_P:.0f}%)")
    
    for j, T in enumerate(T_table):
        props = compute_all_properties(P, T)
        
        # Check for failures
        n_nan = sum(1 for k in ['rho', 'u', 's', 'cp', 'cv', 'alpha', 'nabla_ad'] 
                    if np.isnan(props[k]))
        if n_nan > 0:
            n_failed += 1

        else:
            table_data.append({
                'P': P,
                'T': T,
                **props
            })
        
        n_computed += 1

print(f"\nComputation complete:")
print(f"  Total points: {n_computed:,}")
print(f"  Points with NaN values: {n_failed:,} ({100*n_failed/n_computed:.2f}%)")

In [ ]:
def generate_table_header():
    """
    Generate comprehensive header for the lookup table.
    """
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S UTC")
    
    header = f"""\
# ==============================================================================
# PALEOS MgSiO₃ Equation of State Lookup Table
# ==============================================================================
#
# Description:
#   Pre-computed thermodynamic properties for pure MgSiO₃ across the complete
#   planetary mantle phase diagram, suitable for interpolation in planetary
#   interior models.
#
# Generation Info:
#   Generated: {timestamp}
#   Resolution: {TABLE_PPD} points per decade (log-uniform in P and T)
#   Grid size: {TABLE_N_P} x {TABLE_N_T} = {TABLE_N_P * TABLE_N_T:,} points
#              ({n_computed - n_failed:,} valid / {n_failed:,} skipped due to non-convergence)
#
# Domain:
#   Pressure:    [{TABLE_P_MIN:.8e}, {TABLE_P_MAX:.8e}] Pa
#   Temperature: [{TABLE_T_MIN:.8e}, {TABLE_T_MAX:.8e}] K
#
# Interpolation Notes:
#   - Perform bilinear interpolation in (log10(P), log10(T)) space
#   - Points where the EoS failed to converge are omitted from the table;
#     the grid is therefore not strictly regular. See Usage Example below
#     for how to reconstruct the full grid with NaN fill before interpolation.
#   - Expected interpolation error for density: < 1e-5 (relative)
#   - Interpolation across phase boundaries will have larger errors;
#     consider using the phase column to detect and handle these cases
#
# Phase Codes:
#   solid-en    : Orthoenstatite (Pbca), high T and/or low P
#   solid-lpcen : Low-pressure clinoenstatite (P2₁/c), low T
#   solid-hpcen : High-pressure clinoenstatite (C2/c), moderate-to-high P
#   solid-brg   : Bridgmanite (Pbnm), high P
#   solid-ppv   : Post-perovskite (Cmcm), very high P
#   liquid      : Liquid MgSiO₃
#
# EoS Sources:
#   En, LP-CEn, HP-CEn : Sokolova et al. (2022)
#   Bridgmanite        : Wolf et al. (2015)
#   Post-perovskite    : Sakai et al. (2016)
#   Liquid             : Wolf & Bower (2018) [Luo & Deng 2025 params]
#   Phase boundaries   : Sokolova et al. (2022), Ono & Oganov (2005)
#   Melting curve      : Belonoshko et al. (2005) [P < 2.55 GPa],
#                        Fei et al. (2021) [P >= 2.55 GPa]
#
# Columns:
#   1. P         - Pressure [Pa]
#   2. T         - Temperature [K]
#   3. rho       - Density [kg/m³]
#   4. u         - Specific internal energy [J/kg]
#   5. s         - Specific entropy [J/(kg·K)]
#   6. cp        - Isobaric heat capacity [J/(kg·K)]
#   7. cv        - Isochoric heat capacity [J/(kg·K)]
#   8. alpha     - Thermal expansion coefficient [K⁻¹]
#   9. nabla_ad  - Adiabatic gradient (d ln T / d ln P)_S [dimensionless]
#  10. phase     - Phase identifier (see Phase Codes above)
#
# Missing Values:
#   Rows where the EoS failed to converge are omitted entirely.
#   When reconstructing the full grid, fill missing entries with NaN.
#
# Usage Example (Python):
#   import numpy as np
#   from scipy.interpolate import RegularGridInterpolator
#   
#   # Load table (skip header lines starting with #)
#   cols = ['P', 'T', 'rho', 'u', 's', 'cp', 'cv', 'alpha', 'nabla_ad', 'phase']
#   data = np.genfromtxt('mgsio3_eos_table.dat', names=cols, dtype=None,
#                        encoding='utf-8')
#   
#   # Reconstruct the full regular grid (filling gaps with NaN)
#   n_P, n_T = {TABLE_N_P}, {TABLE_N_T}
#   log_P = np.linspace(np.log10({TABLE_P_MIN:.8e}),
#                       np.log10({TABLE_P_MAX:.8e}), n_P)
#   log_T = np.linspace(np.log10({TABLE_T_MIN:.8e}),
#                       np.log10({TABLE_T_MAX:.8e}), n_T)
#   
#   rho = np.full((n_P, n_T), np.nan)
#   for row in data:
#       i = np.searchsorted(log_P, np.log10(row['P']))
#       j = np.searchsorted(log_T, np.log10(row['T']))
#       rho[i, j] = row['rho']
#   
#   # Create interpolator (NaN-aware: will propagate into nearby queries)
#   interp_rho = RegularGridInterpolator((log_P, log_T), rho)
#   
#   # Query at arbitrary P, T
#   P_query, T_query = 50e9, 2500  # 50 GPa, 2500 K
#   rho_interp = interp_rho([[np.log10(P_query), np.log10(T_query)]])[0]
#
# ==============================================================================
"""
    return header

In [ ]:
# Write the table to file
print(f"Writing table to {OUTPUT_FILENAME}...")

with open(OUTPUT_FILENAME, 'w') as f:
    # Write header
    f.write(generate_table_header())
    
    # Column names
    f.write("# P T rho u s cp cv alpha nabla_ad phase\n")
    
    # Data rows
    for row in table_data:
        # Format numeric values in scientific notation
        def fmt(x):
            if np.isnan(x):
                return "NaN".rjust(16)
            else:
                return f"{x:.8e}"
        
        line = (f"{fmt(row['P'])} {fmt(row['T'])} {fmt(row['rho'])} "
                f"{fmt(row['u'])} {fmt(row['s'])} {fmt(row['cp'])} "
                f"{fmt(row['cv'])} {fmt(row['alpha'])} {fmt(row['nabla_ad'])} "
                f"{row['phase']}\n")
        f.write(line)

print(f"Done! Table written to {OUTPUT_FILENAME}")

# Report file size
import os
file_size = os.path.getsize(OUTPUT_FILENAME)
print(f"File size: {file_size / 1e6:.1f} MB")

In [ ]:
# Verification: Quick sanity check of the generated table
print("\nVerification: Reading back table and checking consistency...")
print("=" * 60)

# Read the table
numeric_cols = ['P', 'T', 'rho', 'u', 's', 'cp', 'cv', 'alpha', 'nabla_ad']
data = np.genfromtxt(OUTPUT_FILENAME, names=numeric_cols + ['phase'], dtype=None, encoding='utf-8')

print(f"Table shape: {len(data)} rows")
print(f"Columns: {data.dtype.names}")

# Check for NaN counts
print("\nNaN counts by column:")
for col in numeric_cols:
    n_nan = np.sum(np.isnan(data[col].astype(float)))
    print(f"  {col:10s}: {n_nan:6d} ({100*n_nan/len(data):.2f}%)")

# Phase distribution
phases, counts = np.unique(data['phase'], return_counts=True)
print("\nPhase distribution:")
for phase, count in zip(phases, counts):
    print(f"  {phase:20s}: {count:6d} ({100*count/len(data):.1f}%)")

# Value ranges
print("\nValue ranges (excluding NaN):")
for col in ['rho', 'u', 's', 'cp', 'cv', 'alpha', 'nabla_ad']:
    vals = data[col].astype(float)
    valid = vals[~np.isnan(vals)]
    if len(valid) > 0:
        print(f"  {col:10s}: [{valid.min():.3e}, {valid.max():.3e}]")